# MicroBench — paired WASM vs. native operation timings

This notebook runs the same tightly-scoped operations natively (mamba env, Binder,
local Jupyter) and inside JupyterLite (browser WebAssembly).  Each operation is
timed on the same MiniSEED input, best-of-5 internally, and the per-operation
best times are dumped to `benchmark-results/micro_bench_<run>.json` alongside
the regular `benchmark-results/*.csv` written by `seismo_bench`.

The point is decomposition: comparing overall notebook wall time across
environments only tells us "WASM is slower".  This notebook tells us *which*
piece is slower and by how much (filesystem read, ObsPy path-based open, format
auto-detection, decode, scipy.signal filters, numpy inner loops).

Operations measured:

1. `open(path,'rb').read()`               — raw filesystem read of the ~12 MB MSEED
2. `obspy.read(path)`                     — path-based read *with* format auto-detection
3. `obspy.read(path, format="MSEED")`     — path-based read *without* auto-detection
4. `obspy.read(BytesIO(buf), format=...)` — in-memory read; isolates decode-only
5. `stream.detrend('linear')`             — numpy inner loop
6. `stream.filter('bandpass', zerophase=True)` — scipy.signal SOS filter
7. `np.correlate(a, b, 'same')`           — the `correlateNoise` inner loop (7200-sample windows)
8. `np.fromfile(f, dtype=np.int32, count=N)` — numpy IO baseline


In [ ]:
# stdlib imports only — should be near-free on both native and WASM.
import sys
import platform as _platform
import time
import statistics
import json as _json
import os
import random as _random
import shutil
import tarfile
import zipfile
from io import BytesIO
from pathlib import Path


In [ ]:
# Cold import of numpy (C extension loading, big binding surface).
import numpy as np
print("numpy   :", np.__version__)


In [ ]:
# Cold import of scipy.signal (pulls in the rest of scipy on first use).
import scipy.signal
import scipy
print("scipy   :", scipy.__version__)


In [ ]:
# Cold import of obspy — pulls in the rest of the ObsPy package tree, but
# will NOT re-import numpy/scipy which are already cached above.  So this
# cell measures the *incremental* cost of adding ObsPy on top of the SciPy
# stack, not the whole scientific import.
import obspy
print("obspy   :", obspy.__version__)


In [ ]:
# JupyterLite/emscripten compatibility: replace np.memmap with an in-memory
# read that has identical semantics for how obspy.io.mseed consumes the
# buffer (np.memmap raises OSError 43 on the Emscripten browser FS).
def _is_emscripten():
    return (
        sys.platform == "emscripten"
        or _platform.system() == "Emscripten"
        or "Emscripten" in _platform.platform()
    )

if _is_emscripten():
    def _fake_memmap(filename, dtype=np.uint8, mode=None, offset=0,
                     shape=None, order=None):
        with open(filename, "rb") as f:
            f.seek(offset)
            data = np.fromfile(f, dtype=dtype)
        if shape is not None:
            data = data[:int(np.prod(shape))].reshape(shape, order=order or "C")
        return data
    np.memmap = _fake_memmap

print("python  :", sys.version.split()[0])
print("platform:", sys.platform, "|", _platform.platform())


In [ ]:
DATA_PATH = "../Noise/data/noise.CI.MLAC.LHZ.2004.294.2005.017.mseed"
N_REPS = 5   # best-of-N per operation

# Results collector — flushed to disk in the final cell.
_MICRO = {
    "environment": os.environ.get("SEISMO_BENCH_ENV", "unset"),
    "browser":     os.environ.get("SEISMO_BENCH_BROWSER", "unset"),
    "device":      os.environ.get("SEISMO_BENCH_DEVICE", "unset"),
    "run_index":   int(os.environ.get("SEISMO_BENCH_RUN_INDEX", "0")),
    "python":      sys.version.split()[0],
    "obspy":       obspy.__version__,
    "numpy":       np.__version__,
    "platform":    _platform.platform(),
    "ops": [],
}

def bench(name, fn, n=N_REPS, setup=None):
    """Run fn() n times, best/mean/stdev in seconds. Optional setup() runs before each rep."""
    samples = []
    result = None
    for _ in range(n):
        if setup is not None:
            setup()
        t0 = time.perf_counter()
        result = fn()
        samples.append(time.perf_counter() - t0)
    best = min(samples)
    mean = statistics.fmean(samples)
    stdev = statistics.stdev(samples) if n > 1 else 0.0
    row = {"name": name, "n": n,
           "best_s": best, "mean_s": mean, "stdev_s": stdev,
           "all_s": samples}
    _MICRO["ops"].append(row)
    print(f"{name:<32s}  best={best*1000:8.2f} ms   mean={mean*1000:8.2f} ms   "
          f"stdev={stdev*1000:6.2f} ms   n={n}")
    return result


In [ ]:
# Load bytes once (cost NOT counted — used by later BytesIO ops).
with open(DATA_PATH, "rb") as _f:
    _BUF = _f.read()
_FILE_SIZE_MB = len(_BUF) / 1024 / 1024
print(f"file size: {_FILE_SIZE_MB:.2f} MB ({len(_BUF)} bytes)")

# Reference stream (used by detrend/bandpass/correlate).
_ST_REF = obspy.read(BytesIO(_BUF), format="MSEED")
_TR_REF = _ST_REF.merge(method=1, fill_value=0)[0]
_N_SAMPLES = _TR_REF.stats.npts
_SR = _TR_REF.stats.sampling_rate
print(f"traces after merge : 1  |  npts: {_N_SAMPLES}  |  sr: {_SR} Hz")

_MICRO["file_size_bytes"] = len(_BUF)
_MICRO["decoded_samples"] = int(_N_SAMPLES)
_MICRO["sampling_rate_hz"] = float(_SR)


In [ ]:
def _op_read_bytes():
    with open(DATA_PATH, "rb") as f:
        return f.read()

buf = bench("read_bytes", _op_read_bytes)
assert len(buf) == len(_BUF)


In [ ]:
def _op_obspy_read_path():
    return obspy.read(DATA_PATH)

st = bench("obspy_read_path", _op_obspy_read_path)
print(f"  -> traces={len(st)}, npts_total={sum(tr.stats.npts for tr in st)}")


In [ ]:
def _op_obspy_read_path_format():
    return obspy.read(DATA_PATH, format="MSEED")

st = bench("obspy_read_path_format", _op_obspy_read_path_format)
print(f"  -> traces={len(st)}, npts_total={sum(tr.stats.npts for tr in st)}")


In [ ]:
def _op_obspy_read_bytesio():
    return obspy.read(BytesIO(_BUF), format="MSEED")

st = bench("obspy_read_bytesio", _op_obspy_read_bytesio)
print(f"  -> traces={len(st)}, npts_total={sum(tr.stats.npts for tr in st)}")


In [ ]:
# Same as obspy_read_path_format but skips the tarfile.is_tarfile + zipfile.is_zipfile
# probes inside the uncompress_file decorator (obspy/core/util/decorator.py).
def _op_obspy_read_path_nocompress():
    return obspy.read(DATA_PATH, format="MSEED", check_compression=False)

st = bench("obspy_read_path_nocompress", _op_obspy_read_path_nocompress)
print(f"  -> traces={len(st)}, npts_total={sum(tr.stats.npts for tr in st)}")


In [ ]:
# Isolate the tarfile.is_tarfile(path) cost — a suspected chunk of the extra
# WASM overhead on the path branch.
def _op_tarfile_probe():
    return tarfile.is_tarfile(DATA_PATH)

bench("tarfile_is_tarfile", _op_tarfile_probe)


In [ ]:
# Isolate the zipfile.is_zipfile(path) cost — opens the file, seeks to near the
# end, reads up to ~64 KB looking for the ZIP End-Of-Central-Directory record.
# Interesting under WASM because it exercises seek+random-access reads.
def _op_zipfile_probe():
    return zipfile.is_zipfile(DATA_PATH)

bench("zipfile_is_zipfile", _op_zipfile_probe)


In [ ]:
# Read a fixed 1 MB from the file in progressively smaller chunk sizes.
# Native FS: throughput roughly independent of chunk size (page cache serves
# small reads too).  Emscripten FS: cost should grow roughly linearly with
# the number of read() syscalls once chunks are below the FS's block size.
# This directly pins down "how expensive is one read() syscall in WASM".
def _read_chunks(chunk_size, total_bytes=1024*1024):
    got = 0
    with open(DATA_PATH, "rb") as f:
        while got < total_bytes:
            b = f.read(chunk_size)
            if not b:
                break
            got += len(b)
    return got

for _cs in (1024*1024, 65536, 4096, 512):
    _label = f"open_read_1MB_chunk{_cs:>7d}B"
    bench(_label, lambda _cs=_cs: _read_chunks(_cs))


In [ ]:
# Pure stat syscall cost.  Each rep does 1000 stats, we divide by 1000 in
# the JSON post-processing (bench() reports whole-cell timings — the raw
# 'best_s' will be ~1000× a single stat).  Native ~microseconds; WASM
# expected to be dominated by the WASM<->JS boundary crossing.
_N_STAT = 200   # keep small — 200 * ~2 ms/stat WASM = ~0.4 s per rep

def _op_stat_batch():
    for _ in range(_N_STAT):
        os.stat(DATA_PATH)

bench(f"os_stat_x{_N_STAT}", _op_stat_batch)


In [ ]:
# os.getpid() does not touch the filesystem — under Emscripten it is still
# a syscall that has to cross the WASM<->JS boundary.  If this shows the
# same ~2 ms/call as os.stat, the whole 'FS is slow' story is really 'the
# boundary crossing is slow' and every syscall pays the same tax.
_N_BOUNDARY = 200   # 200 * ~2 ms WASM = ~0.4 s per rep

def _op_bare_boundary():
    for _ in range(_N_BOUNDARY):
        os.getpid()

bench(f"bare_boundary_getpid_x{_N_BOUNDARY}", _op_bare_boundary)


In [ ]:
# Cost of opening a file, without any read work.  Compared to open+read,
# tells us how much is 'open()' vs how much is 'read()'.
_N_OPEN = 30    # 30 * ~5 ms WASM open/close = ~0.15 s per rep

def _op_open_close_only():
    for _ in range(_N_OPEN):
        f = open(DATA_PATH, "rb")
        f.close()

bench(f"fs_open_close_only_x{_N_OPEN}", _op_open_close_only)


In [ ]:
# Reads the first 1 MB in 4 KB chunks through a SINGLE open() — so every
# iteration cost is one read() + implicit sequential seek, no open.  Same
# byte budget as fs_chunk_sweep's 4 KB entry, but with only one open() —
# so subtracting the two isolates the open() cost.
def _op_reads_on_open_handle():
    n_reads = 0
    with open(DATA_PATH, "rb") as f:
        while n_reads * 4096 < 1024*1024:
            b = f.read(4096)
            if not b:
                break
            n_reads += 1
    return n_reads

bench("fs_reads_4k_single_open_1MB", _op_reads_on_open_handle)


In [ ]:
# 200 iterations of (seek to random offset in first 1 MB, read 64 bytes).
# Mimics what zipfile.is_zipfile does under the hood.  Isolates seek+small-read
# cost from open cost.
_N_SEEK = 50    # 50 * (seek + tiny read) ~ still enough resolution

def _op_seek_read_small():
    with open(DATA_PATH, "rb") as f:
        for _ in range(_N_SEEK):
            f.seek(_random.randrange(0, 1024*1024 - 64))
            f.read(64)

bench(f"fs_seek_read_64B_x{_N_SEEK}", _op_seek_read_small)


In [ ]:
# detrend is in-place; give each rep its own copy.
_st_work = None
def _setup_detrend():
    global _st_work
    _st_work = _ST_REF.copy()

def _op_detrend():
    _st_work.detrend("linear")
    return _st_work

bench("stream_detrend_linear", _op_detrend, setup=_setup_detrend)


In [ ]:
_st_work = None
def _setup_bandpass():
    global _st_work
    _st_work = _ST_REF.copy()

def _op_bandpass():
    _st_work.filter("bandpass", freqmin=0.1, freqmax=0.2, corners=4, zerophase=True)
    return _st_work

bench("stream_bandpass_zerophase", _op_bandpass, setup=_setup_bandpass)


In [ ]:
# Same numerical work as stream_detrend_linear, but without ObsPy's
# Stream/Trace wrapper — so we can tell whether the WASM slowdown is inside
# scipy's C code or in ObsPy's Python-level bookkeeping around the call.
_arr_ref = _TR_REF.data.astype(np.float64, copy=True)

_arr_work = None
def _setup_scipy_detrend():
    global _arr_work
    _arr_work = _arr_ref.copy()

def _op_scipy_detrend():
    return scipy.signal.detrend(_arr_work, type="linear")

bench("scipy_detrend_direct", _op_scipy_detrend, setup=_setup_scipy_detrend)


In [ ]:
# np.correlate at 100 / 7200 / 72000 samples ('same' mode, float64).
# 7200 matches the correlateNoise inner loop window; the other two flank it.
# If the WASM/native ratio is roughly constant across sizes, it is a pure
# per-element WASM cost.  If it blows up at one size, that is either SIMD
# activation, cache boundary, or an algorithmic threshold to investigate.
def _make_pair(n):
    a = _TR_REF.data[:n].astype(np.float64, copy=True)
    b = _TR_REF.data[n:2*n].astype(np.float64, copy=True)
    if len(a) < n: a = np.pad(a, (0, n - len(a)))
    if len(b) < n: b = np.pad(b, (0, n - len(b)))
    return a, b

for _n in (100, 7200, 72000):
    _a, _b = _make_pair(_n)
    bench(f"np_correlate_{_n:>6d}_same",
          lambda a=_a, b=_b: np.correlate(a, b, mode="same"))


In [ ]:
# Read the file as raw int32 with numpy (no MSEED parsing) — a filesystem +
# numpy IO baseline directly comparable to the pure-bytes read.
def _op_np_fromfile():
    with open(DATA_PATH, "rb") as f:
        return np.fromfile(f, dtype=np.int32)

arr = bench("np_fromfile_int32", _op_np_fromfile)
print(f"  -> len={len(arr)}, dtype={arr.dtype}")


In [ ]:
# Under xeus-python /tmp is typically a MEMFS mount (in-heap linear memory),
# whereas the notebook working directory is served by a different backend
# (likely a lazy-loading IndexedDB / fetch-cached shim).  If that model is
# right, /tmp opens are near-native while notebook-mount opens pay the ~1.4 s
# per-open tax we measured elsewhere in this notebook.

TMP_PATH = "/tmp/noise_micro_bench.mseed"

# (a) Cost of copying source -> /tmp.  Includes the slow src open() so this
#     is NOT a pure /tmp write measurement; useful as an upper bound.
def _setup_tmp_copy():
    if os.path.exists(TMP_PATH):
        os.remove(TMP_PATH)

def _op_tmp_copy_from_src():
    shutil.copy(DATA_PATH, TMP_PATH)

bench("tmp_shutil_copy_src_to_tmp", _op_tmp_copy_from_src,
      setup=_setup_tmp_copy)

# Ensure the file exists for the following probes (the last bench rep left it).
if not os.path.exists(TMP_PATH):
    shutil.copy(DATA_PATH, TMP_PATH)

# (b) Bulk read from /tmp — expected ~microseconds if MEMFS.
def _op_tmp_read_bytes():
    with open(TMP_PATH, "rb") as f:
        return f.read()

bench("tmp_read_bytes", _op_tmp_read_bytes)

# (c) 30 opens+closes on /tmp — direct comparison to fs_open_close_only_x30.
_N_TMP_OPEN = 30

def _op_tmp_open_close():
    for _ in range(_N_TMP_OPEN):
        f = open(TMP_PATH, "rb")
        f.close()

bench(f"tmp_fs_open_close_x{_N_TMP_OPEN}", _op_tmp_open_close)

# (d) np.fromfile from /tmp.
def _op_tmp_np_fromfile():
    with open(TMP_PATH, "rb") as f:
        return np.fromfile(f, dtype=np.int32)

bench("tmp_np_fromfile", _op_tmp_np_fromfile)

# (e) End-to-end: obspy.read from /tmp with the compression-check disabled.
#     Should approach native speed if /tmp is MEMFS.
def _op_tmp_obspy_read_path_nocompress():
    return obspy.read(TMP_PATH, format="MSEED", check_compression=False)

bench("tmp_obspy_read_path_nocompress", _op_tmp_obspy_read_path_nocompress)

# Best-effort emscripten FS introspection (works if the js module is available
# and exposes the FS global; xeus-python often does not).
try:
    import js
    fs = getattr(js.globalThis, "FS", None)
    if fs is None:
        print("(no globalThis.FS in this xeus-python build — cannot inspect mount types)")
    else:
        for _p in ("/", "/tmp", "/home", "/drive"):
            try:
                _mt = fs.lookupPath(_p).node.mount.type.name
                print(f"  mount at {_p:8s}: {_mt}")
            except Exception as _e:
                print(f"  {_p}: {type(_e).__name__}: {_e}")
except Exception as _e:
    print("FS introspection unavailable:", _e)

# Cleanup — don't leave state in the browser FS.
try:
    os.remove(TMP_PATH)
except FileNotFoundError:
    pass


In [ ]:
# The 'plotting/rendering' phase's WASM/native ratio (~4x) has been unexplained.
# The cell that gets timed under that phase does two things: print ~20 lines and
# json.dump.  Measure them separately.

def _op_print_20_lines():
    for i in range(20):
        print(f"probe line {i:2d}: benchmark result placeholder")

def _op_json_dump():
    Path("benchmark-results").mkdir(exist_ok=True)
    with open("benchmark-results/_scratch_probe.json", "w") as fh:
        _json.dump(_MICRO, fh, indent=2)

bench("plot_print_20_lines", _op_print_20_lines)
bench("plot_json_dump_micro", _op_json_dump)


In [ ]:
# Summary table + write JSON alongside the standard seismo_bench outputs.
_MICRO["throughput_MBps"] = {}
for row in _MICRO["ops"]:
    if row["name"] in ("read_bytes", "np_fromfile_int32"):
        _MICRO["throughput_MBps"][row["name"]] = _FILE_SIZE_MB / row["best_s"]

print()
print(f"{'operation':<32s} {'best (ms)':>12s} {'mean (ms)':>12s} {'stdev (ms)':>12s}")
print("-" * 72)
for row in _MICRO["ops"]:
    print(f"{row['name']:<32s} "
          f"{row['best_s']*1000:>12.2f} "
          f"{row['mean_s']*1000:>12.2f} "
          f"{row['stdev_s']*1000:>12.2f}")

from pathlib import Path as _Path
_out_dir = _Path("benchmark-results")
_out_dir.mkdir(parents=True, exist_ok=True)
_stem = (f"micro_bench__env-{_MICRO['environment']}"
         f"__dev-{_MICRO['device']}__run-{_MICRO['run_index']}.json")
_out_path = _out_dir / _stem
with _out_path.open("w") as _f:
    _json.dump(_MICRO, _f, indent=2)
print()
print("wrote", _out_path)
